# Baseline

## Step 1: Load the data and split it

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

from churn import config

In [3]:
df = pd.read_parquet(config.CLEAN_DATA_FILE)
X = df.drop(columns=[config.TARGET])
y = df[config.TARGET]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=config.RANDOM_STATE
)

In [5]:
print(X_train.shape, X_test.shape)
print(f"churn rate  train={y_train.mean():.4f}  test={y_test.mean():.4f}")

(5634, 20) (1409, 20)
churn rate  train=0.2654  test=0.2654


### Step 2: The full Pipeline

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from churn.features import add_features, build_preprocessor, collapse_no_internet

In [7]:
def build_pipeline(model, model_type, num_cols=None, cat_cols=None):
    """Raw rows in → churn prediction out. model_type: "lr" or "tree" (picks the preprocessor)."""
    return Pipeline([
        ("collapse", FunctionTransformer(collapse_no_internet)),
        ("features", FunctionTransformer(add_features)),
        ("prep", build_preprocessor(model_type, num_cols, cat_cols)),
        ("model", model),
    ])

In [8]:
from sklearn.linear_model import LogisticRegression

pipe = build_pipeline(LogisticRegression(max_iter=1000), "lr").fit(X_train, y_train)
pipe[:-1].transform(X_train.head()).shape

(5, 29)

### Step 3: CV setup and the Dummy floor

In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

Why a Dummy model: it predicts the majority class ("no churn") for everyone. It sets the floor that every real model must beat, and it shows why accuracy is misleading: it scores 73.5% accuracy while catching zero churners. Its PR-AUC equals the churn rate (about 0.265), which is the PR-AUC baseline.

cross_validate with a scoring dict computes all the metrics in one run. "average_precision" is sklearn's name for PR-AUC

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)
SCORING = {
    "roc_auc": "roc_auc", "pr_auc": "average_precision",
    "precision": "precision", "recall": "recall", "f1": "f1", "accuracy": "accuracy",
}

StratifiedKFold(5, shuffle=True) splits train into 5 folds that each keep the 26.5% churn rate. Each fold takes one turn as validation. The mean shows how good the model is, and the std shows how stable that score is

In [11]:
def cv_scores(pipe, name):
    """Mean of each metric over the 5 folds, plus the std of ROC-AUC and PR-AUC."""
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    row = {m: res[f"test_{m}"].mean() for m in SCORING}
    row["roc_auc_std"] = res["test_roc_auc"].std()
    row["pr_auc_std"] = res["test_pr_auc"].std()
    return pd.Series(row, name=name)

In [12]:
results = [
    cv_scores(build_pipeline(DummyClassifier(strategy="prior"), "lr"), "dummy"),
    cv_scores(build_pipeline(LogisticRegression(max_iter=1000), "lr"), "lr"),
]
pd.DataFrame(results).round(3)

,roc_auc,pr_auc,precision,recall,f1,accuracy,roc_auc_std,pr_auc_std
dummy,0.500,0.265,0.000,0.000,0.000,0.735,0.000,0.000
lr,0.848,0.663,0.675,0.539,0.599,0.809,0.011,0.013


- dummy is the floor. ROC-AUC 0.5 is coin-flip ranking, and PR-AUC 0.265 equals the churn rate.
- lr ranks customers well (0.847), but at the 0.5 threshold it catches only 54% of churners.
- The std columns (0.011 and 0.013) are small, so LR scores about the same on every fold.

Two things to think about from your results:
1. LR gets about 0.81 accuracy and the Dummy about 0.735. Why is that gap much less impressive than the gap in PR-AUC?

Accuracy's floor is high because of the imbalance. A model that predicts "no churn" for everyone already gets 73.5%, because 73.5% of customers don't churn. Going from 0.735 to 0.81 means fixing only about 28% of the errors left to fix ((0.81 − 0.735) / (1 − 0.735)). Accuracy also doesn't show which errors remain. LR misses about 46% of churners (recall 0.54), and those are the customers the business cares about.

PR-AUC is measured only on the positive class: "when I flag customers, how many really churn, across all levels of recall?" Its floor is the churn rate, 0.265, which is what random guessing gets. LR's 0.66 is about 2.5× that floor. The Dummy can't make PR-AUC look good by ignoring churners the way it does with accuracy.

> In an interview: "With 26.5% positives, a do-nothing model has 73.5% accuracy but catches zero churners, so accuracy rewards ignoring the class we care about. PR-AUC's baseline is the prevalence (0.265), so it measures skill on the churners themselves."



2. Why is ROC-AUC unaffected by the 0.5 threshold, while recall is?

- Recall is computed from hard labels. First you turn probabilities into yes/no with a cutoff (0.5 by default), then you count: churners flagged / all churners. Change the cutoff and the labels change, so recall changes. At 0.3, more customers are flagged, so recall goes up and precision usually goes down.
- ROC-AUC uses the probabilities directly. It tries every possible threshold, traces the true-positive rate against the false-positive rate, and takes the area under that curve. Equivalently, it is the probability that a randomly chosen churner gets a higher score than a randomly chosen non-churner. It depends only on how the customers are ranked, not on where you draw the line.

This matters for the project in two ways:
- LR's ROC-AUC of 0.85 says the model ranks customers well. The low recall (0.54) is mostly because 0.5 is the wrong cutoff for this problem, not because the model is weak.
- That is why the spec separates the two decisions. You choose the model by ROC-AUC/PR-AUC (ranking quality, step 8), then choose the threshold separately from out-of-fold predictions (step 7), based on the business cost: missing a churner costs more than an unnecessary retention offer.

Also, class_weight="balanced" (step 4) mostly shifts the probabilities upward, which has a similar effect to lowering the threshold. So expect recall to jump while ROC-AUC barely moves. Check that when you run step 4.


### Step 4: Baselines with and without imbalance handling

- `class_weight="balanced"` (LR, DT): errors on a churner cost about 2.8× more in the loss (non-churners / churners). The model then pushes churn probabilities up, so recall goes up and precision goes down.
- `scale_pos_weight = neg / pos` (XGBoost): the same idea, using XGBoost's own parameter. It's computed from y_train. In theory each fold should compute its own ratio, but it's a class count that is about 2.77 in every stratified fold, so the effect is negligible. Say that if you're asked.
- Why no SMOTE: weighting does the same job without inventing synthetic customers, and it is simpler to defend.
- Train vs validation score (`return_train_score=True`): this is how you spot overfitting. If train AUC is about 1.0 and validation AUC is much lower, the model memorised the training folds.
- `n_jobs=1` inside XGBoost: cross_validate already runs the 5 folds in parallel (`n_jobs=-1`). If XGBoost also used every core, the threads would compete for the CPU and run slower.

In [13]:
def cv_scores(pipe, name):
    """Mean of each metric over the 5 folds, plus stds and the train ROC-AUC (overfitting check)."""
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1,
                         return_train_score=True)
    row = {m: res[f"test_{m}"].mean() for m in SCORING}
    row["roc_auc_std"] = res["test_roc_auc"].std()
    row["pr_auc_std"] = res["test_pr_auc"].std()
    row["train_roc_auc"] = res["train_roc_auc"].mean()
    return pd.Series(row, name=name)

In [14]:
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

spw = (y_train == 0).sum() / (y_train == 1).sum()   # ≈ 2.77
RS = config.RANDOM_STATE

models = {
    "dummy":      (DummyClassifier(strategy="prior"), "lr"),
    "lr":         (LogisticRegression(max_iter=1000), "lr"),
    "lr_bal":     (LogisticRegression(max_iter=1000, class_weight="balanced"), "lr"),
    "dt":         (DecisionTreeClassifier(random_state=RS), "tree"),
    "dt_bal":     (DecisionTreeClassifier(random_state=RS, class_weight="balanced"), "tree"),
    "dt_d5_bal":  (DecisionTreeClassifier(max_depth=5, random_state=RS, class_weight="balanced"), "tree"),
    "xgb":        (XGBClassifier(random_state=RS, n_jobs=1), "tree"),
    "xgb_bal":    (XGBClassifier(random_state=RS, n_jobs=1, scale_pos_weight=spw), "tree"),
}

baseline = pd.DataFrame([cv_scores(build_pipeline(m, t), name) for name, (m, t) in models.items()])
baseline.round(3).sort_values("roc_auc", ascending=False)

,roc_auc,pr_auc,precision,recall,f1,accuracy,roc_auc_std,pr_auc_std,train_roc_auc
lr,0.848,0.663,0.675,0.539,0.599,0.809,0.011,0.013,0.852
lr_bal,0.848,0.662,0.520,0.796,0.629,0.751,0.011,0.014,0.852
dt_d5_bal,0.827,0.610,0.492,0.798,0.609,0.728,0.012,0.028,0.853
xgb_bal,0.824,0.624,0.553,0.668,0.605,0.768,0.009,0.018,0.990
xgb,0.823,0.628,0.627,0.514,0.565,0.790,0.009,0.014,0.985
dt_bal,0.659,0.384,0.501,0.498,0.499,0.735,0.015,0.017,1.000
dt,0.654,0.377,0.490,0.488,0.489,0.729,0.015,0.015,1.000
dummy,0.500,0.265,0.000,0.000,0.000,0.735,0.000,0.000,0.500


LR is the strongest baseline; trees overfit with default settings; balancing changes recall, not AUC.

- `dt_d5_bal` has the highest `pr_auc_std` (0.028), about twice the others. A single shallow tree is unstable: small changes in the data change its splits. That's one reason ensembles like XGBoost exist.
- `dt_d5_bal`'s train AUC is 0.853 against 0.827 on validation. Limiting the depth brought the gap down from 0.34 to 0.03, so the overfitting is mostly gone. A single tree still can't rank as well as LR.

dt_d5_bal isn't in the spec. I added it as a quick contrast to show what a single limit on the tree (max_depth=5) does, before Optuna tunes it properly.

##### Questions to answer from your table:
1. For lr vs lr_bal, which metrics moved, and which stayed about the same? Link it back to answer 2 from before.

|  | ROC-AUC | PR-AUC | Precision | Recall | F1 | Accuracy |
| --- | --- | --- | --- | --- | --- | --- |
| `lr` | 0.847 | 0.664 | 0.675 | 0.537 | 0.598 | 0.808 |
| `lr_bal` | 0.847 | 0.662 | 0.522 | 0.796 | 0.630 | 0.752 |

* **Stayed the same:** ROC-AUC and PR-AUC. These measure how well the model ranks customers, and the ranking hardly changed.


* **Moved:** every metric that depends on the threshold. Recall rose $0.54 \to 0.80$, precision fell $0.68 \to 0.52$, and accuracy fell because the model now flags more non-churners.


* **Why:** weighting churners about $2.8\times$ mainly raises the intercept, so every customer's churn probability goes up by a similar amount. Customers keep roughly the same order, so the AUCs don't change. More customers now cross $0.5$, which is the same effect as lowering the threshold (answer 2 from before).


* **What it means for the project:** for LR, `class_weight="balanced"` and picking a lower threshold are almost interchangeable. One difference matters for the app: the balanced model's probabilities are **no longer calibrated**. A score of $0.6$ doesn't mean a $60\%$ chance of churning. If the Streamlit app shows "churn probability: $62\%$", the unweighted model with a tuned threshold gives a more honest number. Keep this in mind for steps 7–8.


2. Why does class_weight change almost nothing for the unlimited tree (dt vs dt_bal)? Hint: what probabilities does a fully grown tree output?

A fully grown tree keeps splitting until almost every leaf is **pure** (only churners or only non-churners). The probability a leaf predicts is the share of churners in it, so it is almost always **0** or **1**.

* The weights change how splits are scored, so the tree's shape changes a little. But the leaves still end up pure, and a pure leaf predicts the same class whatever the weights. The predictions barely change.


* Weights only matter when leaves are **mixed**. Then they decide whether a leaf with, say, 35% churners predicts churn. That's why the depth-limited `dt_d5_bal` reaches 0.80 recall.


* This also explains the low ROC-AUC (0.66). With almost only 0/1 outputs, the model can't rank customers, so the ROC curve is nearly a single point. Train AUC is 1.0 because the tree memorised the training folds. The fix is to limit its size (`max_depth`, `min_samples_leaf`, `ccp_alpha`), which is exactly the Optuna search space for DT.


3. Untuned XGBoost loses to plain LR here. Is that a fair verdict yet? Look at its train AUC.

**No, not yet.** Compare train and validation AUC:

|  | Train AUC | Validation AUC | Gap |
| --- | --- | --- | --- |
| `lr` | 0.852 | 0.847 | 0.005 |
| `xgb` | 0.987 | 0.821 | 0.166 |

* **XGBoost is overfitting.** Its defaults (`max_depth=6`, `learning_rate=0.3`, no subsampling) are aggressive for 5,600 rows. LR barely overfits.

* Tuning (shallower trees, a lower learning rate, `min_child_weight`, `subsample`, `reg_lambda`) usually brings XGBoost up to about 0.84–0.85. The fair comparison comes after step 6.

* **Why LR does so well here, which is a good interview point:** the EDA showed the signal is mostly simple and additive. Tenure is about linear in log-odds ($R^2$ 0.92), and the strongest drivers are main effects (contract, internet type, payment method, add-on count). Boosted trees win when there are strong interactions and non-linear effects to find, and this dataset doesn't have many. If tuned XGBoost only matches LR, that supports the EDA.

## Step 5: Feature-set comparisons with CV

#### **Concepts**

* **Ablation:** change one thing, keep everything else fixed, and see how the CV score moves. This turns the "to validate with CV" notes from the EDA into decisions you can defend with evidence.

* **This only uses train data and CV.** The test set isn't touched, so choosing features this way doesn't leak.

* **How large a difference matters:** your ROC-AUC std is about 0.011 across folds. Every variant runs on the **same 5 folds** (fixed `random_state`), so the comparison is paired and somewhat more sensitive than the std suggests. Still, a rule of thumb: **a change below about 0.003 is noise**. **When it's a tie, keep the simpler feature set.** Fewer features means a simpler app form, easier SHAP plots and less to explain.

* **Compare ranking metrics only** (ROC-AUC, PR-AUC). The threshold is chosen later, so recall at 0.5 would only add noise here.

* **Use LR as the main judge.** Collinearity and the count-vs-binaries question are LR concerns, and LR is the strongest model so far. XGBoost runs as a second opinion, but it's still untuned, so treat its small differences as noise.

**What each variant tests**

| Variant | Question it answers |
| --- | --- |
| `+TotalCharges` | Was dropping it right? (EDA decision) |
| `-has_family` | Does the merged Partner/Dependents flag add anything? |
| `-tenure_group` | Do the tenure bands capture the first-6-months jump? |
| `count_only` | LR: `num_services` instead of the 5 add-on binaries |
| `binaries_only` | LR: the 5 binaries instead of `num_services` |
| `+dropped` | Put back every dropped column: were all the drops safe? |

Your current default (`config`) contains **both** `num_services` and the 5 add-on binaries. That's the exact-sum collinearity from the EDA. L2 regularisation keeps LR from failing on it, but this step decides which one to keep.

In [15]:
from churn.config import ADDON_COLS

# The Phase 4 starting feature set, written out on purpose: config.NUM_COLS / CAT_COLS now hold the
# *result* of this ablation, so importing them here would change what "base" means on a re-run.
START_NUM = ["tenure", "MonthlyCharges", "num_services"]
START_CAT = ["SeniorCitizen", "Partner", "Dependents", "MultipleLines", "InternetService",
             "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
             "Contract", "PaperlessBilling", "PaymentMethod", "tenure_group", "has_family"]

def without(cols, *drop):
    """Copy of a column list with some columns removed."""
    return [c for c in cols if c not in drop]

feature_sets = {
    "base":          (START_NUM, START_CAT),
    "+TotalCharges": (START_NUM + ["TotalCharges"], START_CAT),
    "-has_family":   (START_NUM, without(START_CAT, "has_family")),
    "-tenure_group": (START_NUM, without(START_CAT, "tenure_group")),
    "count_only":    (START_NUM, without(START_CAT, *ADDON_COLS)),
    "binaries_only": (without(START_NUM, "num_services"), START_CAT),
    "+dropped":      (START_NUM + ["TotalCharges"],
                      START_CAT + ["gender", "PhoneService", "StreamingMovies"]),
}

In [16]:
RANK_SCORING = {"roc_auc": "roc_auc", "pr_auc": "average_precision"}

def ablation(make_model, model_type):
    """CV ROC-AUC / PR-AUC for each feature set, with the change vs 'base'."""
    rows = {}
    for name, (num_cols, cat_cols) in feature_sets.items():
        pipe = build_pipeline(make_model(), model_type, num_cols, cat_cols)
        res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=RANK_SCORING, n_jobs=-1)
        rows[name] = {m: res[f"test_{m}"].mean() for m in RANK_SCORING}
    out = pd.DataFrame(rows).T
    out["d_roc_auc"] = out["roc_auc"] - out.loc["base", "roc_auc"]
    out["d_pr_auc"] = out["pr_auc"] - out.loc["base", "pr_auc"]
    return out.round(4)

ablation(lambda: LogisticRegression(max_iter=1000), "lr")

,roc_auc,pr_auc,d_roc_auc,d_pr_auc
base,0.8474,0.6636,0.0000,0.0000
+TotalCharges,0.8473,0.6636,-0.0000,-0.0000
-has_family,0.8478,0.6636,0.0004,-0.0000
-tenure_group,0.8443,0.6595,-0.0031,-0.0042
count_only,0.8467,0.6616,-0.0007,-0.0021
binaries_only,0.8474,0.6636,-0.0000,0.0000
+dropped,0.8472,0.6646,-0.0001,0.0010


Then the second opinion in its own cell:

In [17]:
ablation(lambda: XGBClassifier(random_state=RS, n_jobs=1, scale_pos_weight=spw), "tree")

,roc_auc,pr_auc,d_roc_auc,d_pr_auc
base,0.8209,0.6188,0.0000,0.0000
+TotalCharges,0.8205,0.6208,-0.0003,0.0020
-has_family,0.8208,0.6191,-0.0001,0.0003
-tenure_group,0.8192,0.6165,-0.0017,-0.0023
count_only,0.8219,0.6154,0.0010,-0.0034
binaries_only,0.8232,0.6263,0.0023,0.0075
+dropped,0.8226,0.6208,0.0017,0.0020


**Why `lambda: ...`:** each feature set needs a **fresh, unfitted** model. The lambda is a small function that creates a new one every time ablation calls `make_model()`.

**Why there's no `TotalCharges` NaN problem:** `collapse_no_internet` and `add_features` keep every column, so `TotalCharges` is still there when the preprocessor selects it. It was filled with `0` for the 11 new customers in `dataset.clean()`.

**I ran it, and expect very small differences** (mostly below 0.003). Only one variant moves LR noticeably. Small differences are a valid result: they show the EDA-based decisions were sound.

### Step 5: answers

Rule used: on the same 5 folds, a change below ~0.003 is noise; when it's a tie, keep the simpler feature set. LR is the main judge (untuned XGBoost is a second opinion only).

**1. Which variant clearly hurts LR? `-tenure_group`.**

| LR | ROC-AUC | PR-AUC |
|---|---|---|
| base | 0.8474 | 0.6636 |
| -tenure_group | 0.8443 (−0.0031) | 0.6595 (−0.0042) |

It's the only change above the noise rule, and XGBoost agrees in direction (−0.0017 / −0.0023). This confirms the EDA log-odds check: tenure is close to linear in log-odds **except the steep first-6-months kink**. Raw `tenure` gives LR one straight line, and it can't bend. The bands (`0-6`, `7-12`, …) give LR a separate coefficient per band, so it can model the jump at the start. Trees can split anywhere on raw `tenure`, so for them the bands matter less. → **Keep `tenure_group`.**

**2. `+TotalCharges` and `+dropped` change almost nothing. Does that support the drops? Yes.**

- `+TotalCharges`: LR −0.0000 / −0.0000. The EDA said `TotalCharges ≈ tenure × MonthlyCharges`; the model agrees that it carries no information beyond those two.
- `+dropped` (TotalCharges + gender + PhoneService + StreamingMovies): LR −0.0001 / +0.0010, both noise.
- **Why "no change" is enough to drop a column:** the burden of proof is on *keeping* a feature. A column that doesn't improve CV still has costs: collinearity (unstable LR coefficients, importance split between correlated columns in SHAP), one more field in the Streamlit form, one more thing that can drift or arrive broken in production, and for `gender` a fairness question with no benefit. Same score with fewer inputs = the better model (Occam's razor).

**3. `count_only` vs `binaries_only` vs `base`: keep `binaries_only` (drop `num_services`).**

| | LR ROC-AUC | LR PR-AUC | XGB ROC-AUC | XGB PR-AUC |
|---|---|---|---|---|
| base (both) | 0.8474 | 0.6636 | 0.8209 | 0.6188 |
| count_only | 0.8467 | 0.6616 | 0.8219 | 0.6154 |
| binaries_only | 0.8474 | 0.6636 | 0.8232 | 0.6263 |

- For LR, `binaries_only` is **identical** to base: `num_services` is an exact sum of the 5 binaries, so LR can already build it (equal weights on the 5 `_Yes` columns). It adds nothing except collinearity. `count_only` is slightly worse: it forces every service to have the same effect.
- XGBoost also scores best with `binaries_only` (+0.0023 / +0.0075, the largest change in its table; still small for an untuned model).
- **Tie-breaker: interpretability.** With the binaries, SHAP says "no **TechSupport**" or "no **OnlineSecurity**", which a retention team can act on (offer that service). The count only says "few services", without saying which ones. The app form asks for the 5 services either way (the count is computed from them), so the count saves the user nothing.

**Also decided:** `-has_family` gives +0.0004 for LR (noise), and the EDA showed that merging Partner and Dependents loses information, so **drop `has_family`**.

### Final feature set (written to `churn.config`)

- `NUM_COLS = ["tenure", "MonthlyCharges"]`. **`num_services` removed.**
- `CAT_COLS`: 14 columns. **`has_family` removed**, `tenure_group` kept.
- Still excluded: `TotalCharges`, `gender`, `PhoneService`, `StreamingMovies` (`config.DROP_COLS`).
- `add_features` still builds all three engineered columns. The two unused ones are dropped by `remainder="drop"`.
- Result: LR 29 columns, trees 38 columns.

The cell below checks that the new config defaults score at least as well as the starting set. **Restart the kernel first** (the config is imported once per kernel session, so an edited `config.py` is only picked up after a restart).

In [18]:
from churn.config import CAT_COLS, NUM_COLS

final_vs_start = {"start (Phase 4)": (START_NUM, START_CAT), "final (config)": (NUM_COLS, CAT_COLS)}

rows = {}
for model_name, make_model, model_type in [
    ("lr", lambda: LogisticRegression(max_iter=1000), "lr"),
    ("xgb_bal", lambda: XGBClassifier(random_state=RS, n_jobs=1, scale_pos_weight=spw), "tree"),
]:
    for set_name, (num_cols, cat_cols) in final_vs_start.items():
        pipe = build_pipeline(make_model(), model_type, num_cols, cat_cols)
        res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=RANK_SCORING, n_jobs=-1)
        rows[(model_name, set_name)] = {m: res[f"test_{m}"].mean() for m in RANK_SCORING}

print("LR columns:", build_pipeline(LogisticRegression(), "lr")[:-1].fit(X_train, y_train).transform(X_train.head()).shape[1])
pd.DataFrame(rows).T.round(4)   # expect lr final ≈ 0.8478 / 0.6635, xgb_bal final ≈ 0.8237 / 0.6245

LR columns: 29


roc_auc  pr_auc
lr      start (Phase 4)   0.8474  0.6636
        final (config)    0.8478  0.6635
xgb_bal start (Phase 4)   0.8209  0.6188
        final (config)    0.8237  0.6245